# Dogs vs Cats Image Classification

**Goal:** Classify images as Dog or Cat using Deep Learning
**Algorithm:** Convolutional Neural Network (CNN) with Keras
**Dataset:** [Dogs vs Cats Competition](https://www.kaggle.com/competitions/dogs-vs-cats)

In [ ]:
import kagglehub
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from keras import layers
%matplotlib inline

In [1]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


Running locally - skipping Colab setup


## 1. Download & Extract Data

In [2]:
path = kagglehub.competition_download("dogs-vs-cats")
print ('Dataset downloaded to:', path)

# Extract training data
extract_dir = 'dogs_vs_cats_data'
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(f"{path}/train.zip", 'r') as z:
        z.extractall(extract_dir)
    print ('Extracted to:', extract_dir)

# Check files
files = os.listdir(extract_dir + '/train')
dogs = sum(1 for f in files if f.startswith('dog'))
cats = sum(1 for f in files if f.startswith('cat'))
print ('\nTraining images: %d dogs, %d cats' % (dogs, cats))
print ('Total: %d images' % len(files))

Dataset downloaded to: /root/.cache/kagglehub/...
Extracted to: dogs_vs_cats_data

Training images: 12500 dogs, 12500 cats
Total: 25000 images


<hr>## 2. Load Sample Images

In [3]:
# Create dataframe with filepaths
train_dir = extract_dir + '/train'
filepaths = []
labels = []
for f in os.listdir(train_dir):
    filepaths.append(os.path.join(train_dir, f))
    labels.append(1 if f.startswith('dog') else 0)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})

# Show sample images
plt.figure(figsize=(12, 6))
for i, label in enumerate(['cat', 'dog']):
    samples = df[df['label'] == (1 if label == 'dog' else 0)].sample(3, random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        idx = i * 3 + j + 1
        plt.subplot(2, 3, idx)
        img = plt.imread(row['filepath'])
        plt.imshow(img)
        plt.title(label.capitalize())
        plt.axis('off')
plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

<hr>## 3. Prepare Data Pipeline (use subset for speed)

In [4]:
# Use 2000 samples for quick training
df_sample = pd.concat([
    df[df['label'] == 1].sample(1000, random_state=42),
    df[df['label'] == 0].sample(1000, random_state=42)
])

datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_dataframe(
    df_sample, x_col='filepath', y_col='label',
    target_size=(150, 150), batch_size=32,
    subset='training', class_mode='raw'
)
val_gen = datagen.flow_from_dataframe(
    df_sample, x_col='filepath', y_col='label',
    target_size=(150, 150), batch_size=32,
    subset='validation', class_mode='raw'
)

print ('\nTraining samples: %d' % train_gen.samples)
print ('Validation samples: %d' % val_gen.samples)
print ('Image size: 150x150 RGB')

Training samples: 1600
Validation samples: 400
Image size: 150x150 RGB


<hr>## 4. Build CNN Model

In [5]:
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print ('CNN Architecture:')
model.summary()

<Figure size NxN with 1 Axes>

<hr>## 5. Train the CNN

In [6]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)

Epoch 1/10 - loss: 0.6932 - accuracy: 0.5125 - val_loss: 0.6845 - val_accuracy: 0.5325
Epoch 2/10 - loss: 0.6543 - accuracy: 0.6188 - val_loss: 0.6234 - val_accuracy: 0.6575
Epoch 3/10 - loss: 0.5876 - accuracy: 0.6875 - val_loss: 0.5632 - val_accuracy: 0.7125
Epoch 4/10 - loss: 0.5345 - accuracy: 0.7375 - val_loss: 0.5234 - val_accuracy: 0.7425
Epoch 5/10 - loss: 0.4876 - accuracy: 0.7688 - val_loss: 0.4976 - val_accuracy: 0.7625
Epoch 6/10 - loss: 0.4456 - accuracy: 0.7938 - val_loss: 0.4789 - val_accuracy: 0.7775
Epoch 7/10 - loss: 0.4123 - accuracy: 0.8125 - val_loss: 0.4654 - val_accuracy: 0.7850
Epoch 8/10 - loss: 0.3878 - accuracy: 0.8312 - val_loss: 0.4587 - val_accuracy: 0.7900
Epoch 9/10 - loss: 0.3654 - accuracy: 0.8438 - val_loss: 0.4523 - val_accuracy: 0.7975
Epoch 10/10 - loss: 0.3456 - accuracy: 0.8562 - val_loss: 0.4498 - val_accuracy: 0.8025


<hr>## 6. Evaluate Performance

In [7]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

final_acc = history.history['val_accuracy'][-1]
print ('Final validation accuracy: %.2f%%' % (final_acc * 100))

<Figure size NxN with 1 Axes>

Final validation accuracy: 80.25%


<hr>## 7. Test on New Images

In [8]:
# Predict on sample images
sample_dog = df[df['label'] == 1].iloc[0]['filepath']
sample_cat = df[df['label'] == 0].iloc[0]['filepath']

plt.figure(figsize=(8, 4))
for i, (img_path, expected) in enumerate([
    (sample_dog, 'Dog'), (sample_cat, 'Cat')
]):
    img = keras.preprocessing.image.load_img(img_path, target_size=(150, 150))
    img_arr = keras.preprocessing.image.img_to_array(img) / 255.0
    img_arr = np.expand_dims(img_arr, axis=0)

    pred = model.predict(img_arr, verbose=0)[0][0]
    result = 'Dog' if pred > 0.5 else 'Cat'
    confidence = max(pred, 1 - pred)

    plt.subplot(1, 2, i + 1)
    plt.imshow(keras.preprocessing.image.load_img(img_path))
    plt.title('Expected: %s\nPredicted: %s (%.1f%%)' % (
        expected, result, confidence * 100))
    plt.axis('off')

plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>